In [1]:
import numpy as np
import pandas as pd
import json
import time
import datetime
import calendar

In [2]:
# load in psiturk data
rm1df = pd.read_json('../../data/db/exported/room1-2.8.19.json', convert_dates=['beginhit','endhit'])
rm2df = pd.read_json('../../data/db/exported/room2-2.8.19.json', convert_dates=['beginhit','endhit'])

# drop runs that didn't finish
rm1df = rm1df[rm1df.status != 1]
rm2df = rm2df[rm2df.status != 1]

# keep relevant columns
rm1df = rm1df[['uniqueid','datastring','beginhit','endhit','hitid','status']].reset_index(drop=True)
rm2df = rm2df[['uniqueid','datastring','beginhit','endhit','hitid','status']].reset_index(drop=True)

# format datastring as dict 
rm1df['datastring'] = rm1df['datastring'].apply(json.loads)
rm2df['datastring'] = rm2df['datastring'].apply(json.loads)

# add test room column
rm1df['testroom'] = 1
rm2df['testroom'] = 2

# remove test runs for each room
rm1df = rm1df.loc[1:].reset_index(drop=True)
rm2df = rm2df.loc[1:].reset_index(drop=True)

# concatenate dataframes
expdf = pd.concat([rm1df, rm2df], ignore_index=True)

In [3]:
# load pre/post questionnaire responses
preqdf = pd.read_csv('../../data/google-form-data/Pre-experiment Questionnaire.csv', parse_dates=[0])
preqdf = preqdf.rename(index=str, columns={'Timestamp':'preqtime'})
postqdf = pd.read_csv('../../data/google-form-data/Post-experiment questionnaire.csv', parse_dates=[0])
postqdf = postqdf.rename(index=str, columns={'Timestamp':'postqtime'})

# exclude test runs
preqdf = preqdf.dropna(subset=['Subject ID'], inplace=False).reset_index(drop=True)
postqdf = postqdf.dropna(subset=['Subject ID'], inplace=False).reset_index(drop=True)

In [4]:
# convert form timestamp to POSIX time **SERIES.APPLY(DT.DT.TIMESTAMP).MULTIPLY(1000) DOES NOT WORK**
newpretimestamp = pd.Series([0]*len(preqdf['preqtime']))
for ix, val in enumerate(newpretimestamp):
    newpretimestamp[ix] = preqdf['preqtime'][ix].timestamp()*1000
preqdf['preqtime'] = newpretimestamp

newposttimestamp = pd.Series([0]*len(postqdf['postqtime']))
for ix, val in enumerate(newposttimestamp):
    newposttimestamp[ix] = postqdf['postqtime'][ix].timestamp()*1000
postqdf['postqtime'] = newposttimestamp

In [5]:
# remove dropped subjects from google form and experiment dfs
dropids = ['MD-102218-B-04','MD-020119-A-01','MD-102318-A-01','MD-101318-A-05','MD-1011318-A-05','MD-020119-B-01',
          'MD-101218-B-04']

for dropid in dropids:
    preqdf = preqdf[preqdf['Subject ID'] != dropid]
    postqdf = postqdf[postqdf['Subject ID'] != dropid]
        
preqdf.reset_index(drop=True, inplace=True)
postqdf.reset_index(drop=True, inplace=True)

# drop a 
expdf = expdf.loc[1:].reset_index(drop=True)

In [6]:
# fix mistaken day/repeat ID assignments

preqdf.at[76,'Subject ID'] = 'MD-102018-A-03'
postqdf.at[76,'Subject ID'] = 'MD-102018-A-03'

preqdf.at[77,'Subject ID'] = 'MD-102218-A-06'
postqdf.at[77,'Subject ID'] = 'MD-102218-A-06'

preqdf.at[85,'Subject ID'] = 'MD-102218-B-06'
postqdf.at[85,'Subject ID'] = 'MD-102218-B-06'

preqdf.at[89,'Subject ID'] = 'MD-013119-A-02'
postqdf.at[90,'Subject ID'] = 'MD-013119-A-02'

### for mapping between Google Forms with experiment IDs and SQLite databases with PsiTurk IDs

In [7]:
# add empty columns from pre/postquestionnaires to expdf
newcols = pd.unique(np.concatenate([i.columns.values for i in [preqdf,postqdf]]))
expdf = pd.concat([expdf, pd.DataFrame(columns=newcols)], sort=False)

turktimes = {}
preqtimes = {}
mapped = []

# get psiturk start times
for ix, sub in expdf.iterrows():
    turktimes[sub['datastring']['data'][0]['dateTime']] = ix
# get gform submit times
for ix, sub in preqdf.iterrows():
    preqtimes[sub['preqtime']] = ix

# for each psiturk time, find row with closest gform submission time
for turktime, rownum in turktimes.items():
    closest = preqtimes.get(turktime, preqtimes[min(preqtimes.keys(), key=lambda k: abs(k-turktime))])
#     print(datetime.datetime.fromtimestamp(turktime/1e3))
#     print(datetime.datetime.fromtimestamp(preqdf.loc[closest]['preqtime'] / 1e3))
#     print('\n\n')
#     print((turktime - preqdf.loc[closest]['preqtime'])/60000 , closest)
#     if closest in mapped:
#         print(str(closest) + ' already there')
#     else:
#         mapped.append(closest)
    
    
    


In [7]:
preqdf

,preqtime,Subject ID,"Outside of this study, have you ever watched an episode of either of the TV shows ""Atlanta"" or ""Arrested Development?""",Is English your first language?,Do you have any hearing or speech impairments?,Do you have normal color vision?,"Are you taking any medications or have you had any recent injuries that could affect your memory or attention? (if so, describe below)","If yes above, describe",In what year were you born?,Sex,Ethnicity,Race (check all that apply),Highest Degree Achieved,"If you are currently an undergraduate, what year are you?",What is/was your major?,How many hours of sleep did you get last night?,How many cups of coffee have you had today?,How alert are you feeling?
0,1539364543000,MD-101218-A-01,I've never watched either one,Yes,No,Yes,No,NaN,2000,Male,Not Hispanic or Latino,White,Some high school,'22,undeclared,7.0,1.0,A little alert
1,1539368333000,MD-101218-B-01,I've never watched either one,Yes,No,Yes,No,NaN,2000,Female,Not Hispanic or Latino,White,High school graduate,'22,undeclared,5.0,0.0,A little sluggish
2,1539368938000,MD-101218-A-02,I've never watched either one,Yes,No,Yes,No,NaN,1999,Male,Not Hispanic or Latino,White,Some college,'22,undeclared,7.0,2.0,A little alert
3,1539372202000,MD-101218-B-02,I've never watched either one,Yes,No,Yes,No,NaN,2000,Female,Not Hispanic or Latino,American Indian or Alaska Native;White,Some college,'22,neuroscience,9.0,0.0,A little alert
4,1539372686000,MD-101218-A-03,I've never watched either one,Yes,No,Yes,No,NaN,2000,Male,Not Hispanic or Latino,Asian,High school graduate,'22,Sociology,7.0,0.0,A little alert
5,1539376148000,MD-101218-B-03,I've never watched either one,Yes,No,Yes,No,NaN,2000,Female,Not Hispanic or Latino,White,Some college,'22,undeclared,8.0,0.0,A little alert
6,1539376344000,MD-101218-A-04,I've never watched either one,Yes,No,Yes,Yes,Concussion (April 2015),1999,Female,Not Hispanic or Latino,White,Some college,'22,Undeclared (but planning on being a psychology...,8.0,1.0,Very sluggish
7,1539451793000,MD-101318-A-01,I've never watched either one,Yes,No,Yes,No,NaN,2000,Female,Not Hispanic or Latino,Black or African American;White,Some college,'22,undeclared,8.0,0.0,A little alert
8,1539452101000,MD-101318-B-01,I've never watched either one,Yes,No,Yes,No,NaN,1999,Female,Not Hispanic or Latino,White,Some college,'22,Undeclared,7.0,2.0,Neutral
9,1539455818000,MD-101318-B-02,I've never watched either one,Yes,No,Yes,No,NaN,1999,NaN,Not Hispanic or Latino,Asian,High school graduate,'22,Psychology,6.0,1.0,Neutral


In [8]:
postqdf

,postqtime,Subject ID,How engaging did you find the episode?,How easy/difficult was it to follow the episode?,How well do you feel you recalled the events of the episode?,How well do you feel you learned the characters' names over the course of the episode?,How tired do you feel?
0,1539367920000,MD-101218-A-01,Very engaging,Somewhat easy,Very well,Very well,A little tired
1,1539371395000,MD-101218-B-01,A little engaging,Somewhat easy,Very well,Somewhat well,A little tired
2,1539371970000,MD-101218-A-02,Very engaging,Somewhat easy,Somewhat well,Somewhat well,A little tired
3,1539374968000,MD-101218-B-02,Very engaging,Very easy,Very well,Somewhat well,A little alert
4,1539375514000,MD-101218-A-03,Very engaging,Somewhat easy,Very well,Somewhat well,Very alert
5,1539378961000,MD-101218-B-03,A little engaging,Somewhat easy,Neutral,Neutral,Neutral
6,1539379335000,MD-101218-A-04,Very engaging,Neutral,Somewhat well,Not very well,A little tired
7,1539454508000,MD-101318-A-01,A little engaging,Neutral,Neutral,Neutral,Neutral
8,1539455459000,MD-101318-B-01,Very engaging,Somewhat easy,Somewhat well,Somewhat well,Very alert
9,1539458649000,MD-101318-A-02,Very engaging,Somewhat easy,Somewhat well,Somewhat well,A little tired


In [9]:
expdf

,uniqueid,datastring,beginhit,endhit,hitid,status,testroom
0,debugBUnNA:debugLtZcs,"{'condition': 0, 'counterbalance': 0, 'assignm...",2018-10-12 19:17:03.503575,2018-10-12 20:10:44.883411,debugTkKFp,3,1
1,debugd1YD1:debug4FrAg,"{'condition': 0, 'counterbalance': 0, 'assignm...",2018-10-12 20:22:07.319435,2018-10-12 21:09:46.566325,debugonOYk,3,1
2,debugGaDml:debugFTHoY,"{'condition': 0, 'counterbalance': 0, 'assignm...",2018-10-12 21:28:01.305584,2018-10-12 22:16:31.057617,debugXHY6O,3,1
3,debugGVTD3:debugfpzCT,"{'condition': 0, 'counterbalance': 0, 'assignm...",2018-10-12 22:59:26.829625,2018-10-13 00:01:24.711451,debuggQ0y6,3,1
4,debugokLIG:debugalG88,"{'condition': 0, 'counterbalance': 0, 'assignm...",2018-10-13 18:28:34.378931,2018-10-13 19:15:12.977411,debugslG65,3,1
5,debugdhnfF:debug6fW93,"{'condition': 0, 'counterbalance': 0, 'assignm...",2018-10-13 19:35:43.889748,2018-10-13 20:32:37.663906,debugszpCm,3,1
6,debugHKLdw:debugk43rK,"{'condition': 0, 'counterbalance': 0, 'assignm...",2018-10-13 20:37:43.034617,2018-10-13 21:27:38.367721,debugHGwAN,3,1
7,debugnj3ww:debugAC3on,"{'condition': 0, 'counterbalance': 0, 'assignm...",2018-10-13 21:34:31.560211,2018-10-13 22:25:03.538624,debughqOmP,3,1
8,debugmRWSb:debugPVXlB,"{'condition': 0, 'counterbalance': 0, 'assignm...",2018-10-13 22:36:59.470450,2018-10-13 23:39:04.416342,debugKyt7W,3,1
9,debug3lixg:debugdwpeu,"{'condition': 0, 'counterbalance': 0, 'assignm...",2018-10-13 23:58:52.853801,2018-10-14 00:49:19.357875,debugNPCaa,3,1
